# tmQM SchNet: charge x position dipole (equivariant)
mu = sum q_i * r_i

In [ ]:
import numpy as np, pandas as pd, time, gzip, os
import matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
from sklearn.metrics import mean_absolute_error, r2_score
from tqdm import tqdm
print('Imports OK')


In [ ]:
df_y = pd.read_csv('./tmQM_y.csv', sep=';')
df_y = df_y.rename(columns={'HL_Gap': 'gap', 'HOMO_Energy': 'homo', 'LUMO_Energy': 'lumo', 'Dipole_M': 'dipole', 'Metal_q': 'metal_charge', 'Electronic_E': 'E_elec', 'Dispersion_E': 'E_disp', 'Polarizability': 'polarizability'})
targets = ['gap', 'homo', 'lumo', 'dipole', 'metal_charge', 'E_elec', 'E_disp', 'polarizability']
print(f'Loaded {len(df_y)} complexes')
df_y[targets].describe()


In [ ]:
import gzip
def atom_to_z(sym):
    tbl = {"H":1,"He":2,"Li":3,"Be":4,"B":5,"C":6,"N":7,"O":8,"F":9,"Ne":10,"Na":11,"Mg":12,"Al":13,"Si":14,"P":15,"S":16,"Cl":17,"Ar":18,"K":19,"Ca":20,"Sc":21,"Ti":22,"V":23,"Cr":24,"Mn":25,"Fe":26,"Co":27,"Ni":28,"Cu":29,"Zn":30,"Br":35,"Ru":44,"Rh":45,"Pd":46,"Ag":47,"Cd":48,"I":53,"Pt":78,"Au":79,"Hg":80,"Mo":42,"W":74,"Ir":77,"Os":76,"Re":75}
    return tbl.get(sym, 0)
def is_TM(z):
    return z in [21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,72,73,74,75,76,77,78,79,80]
def parse_one(xyz_p, bo_p, start_idx=0):
    mols, labels, is_Fe = [], [], []
    with gzip.open(bo_p, "rt") as f: bo_lines = [l.strip() for l in f.readlines()]
    with gzip.open(xyz_p, "rt") as f: xyz_blocks = f.read().strip().split(chr(10))
    bo_blocks, cur = [], []
    for line in bo_lines:
        if line.startswith("CSD_code"):
            if cur: bo_blocks.append(cur); cur = []
        elif line: cur.append(line)
    if cur: bo_blocks.append(cur)
    i, mol_idx = 0, start_idx - 1
    while i < len(xyz_blocks):
        line = xyz_blocks[i].strip(); i += 1
        if not line: continue
        parts = line.split()
        if len(parts)==1 and parts[0].isdigit():
            n_atoms = int(parts[0])
            if i >= len(xyz_blocks): break
            i += 1
            atoms, coords = [], []
            for _ in range(n_atoms):
                if i >= len(xyz_blocks): break
                cline = xyz_blocks[i].strip(); i += 1
                cparts = cline.split()
                if len(cparts) >= 4:
                    try:
                        z = atom_to_z(cparts[0])
                        coords.append([float(x) for x in cparts[1:4]])
                        atoms.append(z)
                    except ValueError: continue
            mol_idx += 1
            if not any(is_TM(z) for z in atoms) or len(atoms) <= 1: continue
            bo_mat = None
            if mol_idx < len(bo_blocks):
                bo_mat = np.zeros((n_atoms, n_atoms), dtype=np.float32)
                for bline in bo_blocks[mol_idx]:
                    bparts = bline.split()
                    try: src_idx = int(bparts[0]) - 1
                    except ValueError: continue
                    j = 2
                    while j+1 < len(bparts):
                        try:
                            nbr = int(bparts[j]) - 1
                            bo_val = float(bparts[j+1])
                            if 0 <= src_idx < n_atoms and 0 <= nbr < n_atoms:
                                bo_mat[src_idx, nbr] = bo_val
                            j += 3
                        except (ValueError, IndexError): j += 1
            mols.append({"atoms": np.array(atoms), "coords": np.array(coords, dtype=np.float32), "bo": bo_mat, "csv_idx": mol_idx})
            labels.append(df_y.iloc[mol_idx][targets].astype(float).values)
            is_Fe.append(float(26 in atoms))
    return mols, labels, is_Fe
all_m, all_l, all_f = [], [], []
for part in [1,2,3]:
    m,l,f = parse_one(f"./tmQM_X{part}.xyz.gz", f"./tmQM_X{part}.BO.gz", start_idx=len(all_m))
    all_m.extend(m); all_l.extend(l); all_f.extend(f)
    print(f"  Part {part}: {len(m)} TM, Fe={int(sum(f))}")
fe_cmplx = all_m
y_fe = np.array(all_l, dtype=np.float32)
is_Fe_arr = np.array(all_f, dtype=np.float32)
print(f"Total: {len(fe_cmplx)} TM, {int(is_Fe_arr.sum())} Fe")


In [ ]:
N_RBF = 20
RBF_SIGMA = 0.1
CUTOFF = 5.0
K_NEIGHBORS = 32
rbf_centers = np.linspace(0.0, CUTOFF, N_RBF)

def build_schnet_graph(atoms, coords):
    N = len(atoms)
    c = torch.tensor(coords, dtype=torch.float32)
    dist = torch.norm(c.unsqueeze(0) - c.unsqueeze(1), dim=2)
    k = min(K_NEIGHBORS + 1, N)
    vals, nbr_idx = torch.topk(dist, k, dim=1, largest=False)
    edge_src, edge_dst, edge_rbf = [], [], []
    for i in range(N):
        for kidx in range(1, k):
            j = nbr_idx[i, kidx].item()
            d = vals[i, kidx].item()
            if d < CUTOFF:
                edge_src.extend([i, j]); edge_dst.extend([j, i])
                rbf = np.exp(-(d - rbf_centers)**2 / (2 * RBF_SIGMA**2))
                edge_rbf.extend([rbf.tolist(), rbf.tolist()])
    ei = torch.zeros(2, 0, dtype=torch.long) if not edge_src else torch.tensor([edge_src, edge_dst], dtype=torch.long)
    ea = torch.zeros(0, N_RBF) if not edge_src else torch.tensor(edge_rbf, dtype=torch.float32)
    return {'z': torch.tensor(atoms, dtype=torch.long), 'pos': torch.tensor(coords, dtype=torch.float32), 'edge_index': ei, 'edge_rbf': ea}

print(f'Building SchNet graphs (K={K_NEIGHBORS}-NN, {N_RBF} RBFs)...')
graphs, y_vals, is_Fe_vals, failed = [], [], [], 0
for i, m in enumerate(tqdm(fe_cmplx, desc='Building')):
    g = build_schnet_graph(m['atoms'], m['coords'])
    if g['edge_index'].shape[1] > 0:
        graphs.append(g); y_vals.append(y_fe[i]); is_Fe_vals.append(is_Fe_arr[i])
    else:
        failed += 1
y_all = torch.tensor(y_vals, dtype=torch.float32)
is_Fe_t = torch.tensor(is_Fe_vals, dtype=torch.float32)
print(f'Built {len(graphs):,} graphs ({failed} failed)')
ec = [g['edge_index'].shape[1] for g in graphs]
print(f'Edges: min={min(ec)}, max={max(ec)}, mean={np.mean(ec):.0f}')
y_mean = y_all.mean(dim=0)
y_std = y_all.std(dim=0).clamp(min=1e-8)
y_norm = (y_all - y_mean) / y_std
idx = np.random.RandomState(42).permutation(len(graphs))
n_tr = int(len(graphs)*0.7); n_va = int(len(graphs)*0.15)
tr_idx = idx[:n_tr]; va_idx = idx[n_tr:n_tr+n_va]; te_idx = idx[n_tr+n_va:]
print(f'Train: {len(tr_idx):,} | Val: {len(va_idx):,} | Test: {len(te_idx):,}')


In [ ]:
HIDDEN = 128
N_LAYERS = 4
N_ATOM_TYPES = 85

def collate_schnet(batch_list):
    z_list, pos_list, ei_list, rbf_list, batch_list2 = [], [], [], [], []
    offset = 0
    for i, g in enumerate(batch_list):
        n = g['z'].size(0)
        z_list.append(g['z']); pos_list.append(g['pos'])
        ei_list.append(g['edge_index'] + offset)
        rbf_list.append(g['edge_rbf'])
        batch_list2.append(torch.full((n,), i, dtype=torch.long, device=g['z'].device))
        offset += n
    return {'z': torch.cat(z_list), 'pos': torch.cat(pos_list), 'edge_index': torch.cat(ei_list, dim=1), 'edge_rbf': torch.cat(rbf_list, dim=0), 'batch': torch.cat(batch_list2), 'n_graphs': len(batch_list)}

class CFConv(nn.Module):
    def __init__(self, hidden, n_rbf):
        super().__init__()
        self.filter_net = nn.Sequential(nn.Linear(n_rbf, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
    def forward(self, h, ei, rbf):
        src, dst = ei[0], ei[1]; N = h.size(0)
        W = self.filter_net(rbf)
        msg = h[src] * W
        out = torch.zeros(N, h.size(1), device=h.device, dtype=h.dtype)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        return out

class SchNet(nn.Module):
    def __init__(self, hidden=HIDDEN, n_layers=N_LAYERS, n_rbf=N_RBF):
        super().__init__()
        self.embed = nn.Embedding(N_ATOM_TYPES, hidden, padding_idx=0)
        self.interactions = nn.ModuleList([CFConv(hidden, n_rbf) for _ in range(n_layers)])
        self.atomwise = nn.ModuleList([nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, hidden)) for _ in range(n_layers)])
        self.scalar_head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden, hidden//2), nn.SiLU(), nn.Linear(hidden//2, 8))
        self.charge_head = nn.Sequential(nn.Linear(hidden, hidden//2), nn.SiLU(), nn.Linear(hidden//2, 1))

    def forward(self, g, batch_mode=False):
        z, pos, ei, rbf = g['z'], g['pos'], g['edge_index'], g['edge_rbf']
        if batch_mode:
            batch = g['batch']; n_graphs = g['n_graphs']
        h = self.embed(z)
        for inter, atom in zip(self.interactions, self.atomwise):
            h = h + atom(inter(h, ei, rbf))
        if batch_mode:
            mean_h = torch.zeros(n_graphs, h.size(1), device=h.device, dtype=h.dtype)
            mean_h.scatter_add_(0, batch.unsqueeze(-1).expand_as(h), h)
            cnt = torch.zeros(n_graphs, device=h.device, dtype=h.dtype)
            cnt.scatter_add_(0, batch, torch.ones_like(batch, dtype=h.dtype))
            mean_h = mean_h / cnt.clamp(min=1).unsqueeze(-1)
            sum_h = mean_h * cnt.unsqueeze(-1)
            h_pool = torch.cat([mean_h, sum_h], dim=1)
            scalar_pred = self.scalar_head(h_pool)
            q = self.charge_head(h).squeeze(-1)
            dipole_mags = torch.zeros(n_graphs, device=h.device)
            for gi in range(n_graphs):
                mask = batch == gi
                qi = q[mask]; pi = pos[mask]
                q_abs_sum = qi.abs().sum() + 1e-8
                r_center = (pi * qi.abs().unsqueeze(-1)).sum(dim=0) / q_abs_sum
                dipole_vec = (qi.unsqueeze(-1) * (pi - r_center)).sum(dim=0)
                dipole_mags[gi] = dipole_vec.norm()
            scalar_pred[:, 3] = dipole_mags
            return scalar_pred, None
        else:
            h_pool = torch.cat([h.mean(dim=0), h.sum(dim=0)], dim=0)
            scalar_pred = self.scalar_head(h_pool)
            q = self.charge_head(h).squeeze(-1)
            q_abs_sum = q.abs().sum() + 1e-8
            r_center = (pos * q.abs().unsqueeze(-1)).sum(dim=0) / q_abs_sum
            dipole_vec = (q.unsqueeze(-1) * (pos - r_center)).sum(dim=0)
            dipole_mag = dipole_vec.norm()
            scalar_pred[3] = dipole_mag
            return scalar_pred, dipole_vec

m = SchNet().to(device)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
g0 = {k: v.to(device) for k, v in graphs[0].items()}; out, dvec = m(g0)
print(f'Single: {list(out.shape)}')
bg0 = {k: v.to(device) for k, v in g0.items()}
bg1 = {k: v.to(device) for k, v in g0.items()}
batched = collate_schnet([bg0, bg1])
out_b, _ = m(batched, batch_mode=True)
print(f'Batch[2]: {list(out_b.shape)}')


In [ ]:
model = SchNet().to(device)
opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=30, factor=0.5)
crit = nn.MSELoss()
y_norm_gpu = y_norm.to(device)
epochs, bs = 100, 64
early_stop_patience = 40
hist = {'tr': [], 'va': [], 'dipole_mae': []}
best_va = float('inf'); best_state = None; patience_counter = 0
print(f'Training {epochs} epochs (batch={bs}) on {device}...')
print(f'Model: SchNet x {N_LAYERS} x {HIDDEN}d')
print(f'Graphs: {len(graphs):,} | RBF: {N_RBF} | K-NN: {K_NEIGHBORS}\n')
for ep in range(epochs):
    model.train(); perm = torch.randperm(len(tr_idx)); tr_loss = 0; n_batches = 0
    for s in range(0, len(tr_idx), bs):
        bidx = tr_idx[perm[s:s+bs]].tolist()
        bg = [{k: v.to(device) for k, v in graphs[i].items()} for i in bidx]
        by = y_norm_gpu[bidx]
        batched = collate_schnet(bg)
        opt.zero_grad()
        preds, _ = model(batched, batch_mode=True)
        loss = crit(preds, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); tr_loss += loss.item(); n_batches += 1
    model.eval()
    with torch.no_grad():
        va_preds = []
        for s in range(0, len(va_idx), bs):
            bidx = va_idx[s:s+bs].tolist()
            bg = [{k: v.to(device) for k, v in graphs[i].items()} for i in bidx]
            batched = collate_schnet(bg)
            preds, _ = model(batched, batch_mode=True)
            va_preds.append(preds)
        vp = torch.cat(va_preds, dim=0)
        va_loss = crit(vp, y_norm_gpu[va_idx])
    sched.step(va_loss)
    hist['tr'].append(tr_loss / max(1, n_batches))
    hist['va'].append(va_loss.item())
    if va_loss < best_va:
        best_va = va_loss; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= early_stop_patience:
        print(f'Early stop at epoch {ep+1}'); break
    if (ep+1) % 10 == 0:
        vp_orig = vp.cpu() * y_std + y_mean
        vt_orig = y_all[va_idx]
        mae_per = torch.mean(torch.abs(vp_orig - vt_orig), dim=0)
        avg_mae = mae_per.mean().item()
        dipole_mae = mae_per[3].item()
        hist['dipole_mae'].append(dipole_mae)
        print(f'Epoch {ep+1:3d} | TrLoss {hist["tr"][-1]:.4f} | VaLoss {va_loss.item():.4f} | AvgMAE {avg_mae:.4f} | DipoleMAE {dipole_mae:.4f} | Pat {patience_counter}')
print(f'\nDone. Best val loss: {best_va:.4f}')
model.load_state_dict(best_state); model.eval()
torch.save(best_state, 'schnet_best.pt')
print('Saved to schnet_best.pt')


In [ ]:
model.eval()
with torch.no_grad():
    te_preds = []
    for s in range(0, len(te_idx), bs):
        bidx = te_idx[s:s+bs].tolist()
        bg = [{k: v.to(device) for k, v in graphs[i].items()} for i in bidx]
        batched = collate_schnet(bg)
        preds, _ = model(batched, batch_mode=True)
        te_preds.append(preds)
    tp = torch.cat(te_preds, dim=0).cpu()
    tp_orig = tp * y_std + y_mean
    tt_orig = y_all[te_idx]
print(f'{"Property":18s} {"MAE":>10s} {"R2":>8s}')
print('-' * 40)
results = []
for i, prop in enumerate(targets):
    yt, yp = tt_orig[:, i].numpy(), tp_orig[:, i].numpy()
    mae = mean_absolute_error(yt, yp); r2 = r2_score(yt, yp)
    print(f'{prop:18s} {mae:10.4f} {r2:8.4f}')
    results.append({'Property': prop, 'MAE': mae, 'R2': r2})
results_df = pd.DataFrame(results)
print(f'\nMean R2: {results_df["R2"].mean():.4f}')
print(f'Dipole R2 (charge x position): {results_df.iloc[3]["R2"]:.4f}')
import matplotlib.pyplot as plt
n_cols = 3
n_rows = int(np.ceil(len(targets) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*6, n_rows*5.5))
axes = axes.flatten()
for ax, i, prop in zip(axes, range(len(targets)), targets):
    yt, yp = tt_orig[:, i].numpy(), tp_orig[:, i].numpy()
    ax.scatter(yt, yp, alpha=0.4, s=5, c='steelblue')
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel(f'True {prop}'); ax.set_ylabel(f'Predicted {prop}')
    ax.set_title(f'{prop} (R2={r2_score(yt, yp):.4f})')
for ax in axes[len(targets):]: ax.set_visible(False)
plt.suptitle(f'SchNet: Charge x Position Dipole', fontsize=15)
plt.tight_layout(); plt.show()
